# 01 — Bronze Ingest: Fleet & Route Performance Analytics
Reads the four raw CSV files from the Lakehouse `Files/raw/` area and writes them as Delta tables in the Bronze schema.

| Table | Source |
|---|---|
| bronze_vehicles | vehicles.csv |
| bronze_routes | routes.csv |
| bronze_deliveries | deliveries.csv |
| bronze_fuel_logs | fuel_logs.csv |

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
LAKEHOUSE_NAME = "transportation_lakehouse"
RAW_PATH       = f"Files/raw"
BRONZE_SCHEMA  = "bronze"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE_NAME}.{BRONZE_SCHEMA}")
print(f"Schema '{BRONZE_SCHEMA}' ready.")

In [ ]:
# ── Upload helper: copy CSV from workspace to Lakehouse Files ────────────────
import os

def upload_if_missing(local_rel, lh_dest):
    """Copy a file from the demo bundle into Lakehouse Files if not already present."""
    dest_full = f"abfss://" + notebookutils.lakehouse.get().abfsPath.split("abfss://")[1].split("/")[0] + "/" + lh_dest
    # Use notebookutils to upload
    notebookutils.fs.cp(local_rel, f"Files/{lh_dest}", overwrite=False)

# Ensure raw folder exists
notebookutils.fs.mkdirs(f"Files/raw")

In [ ]:
# ── vehicles ────────────────────────────────────────────────────────────────
vehicles_schema = StructType([
    StructField("vehicle_id",      StringType(),  False),
    StructField("vehicle_type",    StringType(),  True),
    StructField("depot",           StringType(),  True),
    StructField("capacity_tonnes", DoubleType(),  True),
    StructField("year_registered", IntegerType(), True),
    StructField("status",          StringType(),  True),
    StructField("driver_id",       StringType(),  True),
])

df_vehicles = (
    spark.read
    .option("header", True)
    .schema(vehicles_schema)
    .csv(f"{RAW_PATH}/vehicles.csv")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_vehicles.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.bronze_vehicles")

print(f"bronze_vehicles: {df_vehicles.count():,} rows")

In [ ]:
# ── routes ───────────────────────────────────────────────────────────────────
routes_schema = StructType([
    StructField("route_id",      StringType(), False),
    StructField("origin",        StringType(), True),
    StructField("destination",   StringType(), True),
    StructField("distance_km",   DoubleType(), True),
    StructField("route_type",    StringType(), True),
    StructField("sla_hours",     DoubleType(), True),
    StructField("toll_cost_gbp", DoubleType(), True),
])

df_routes = (
    spark.read.option("header", True).schema(routes_schema)
    .csv(f"{RAW_PATH}/routes.csv")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_routes.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.bronze_routes")

print(f"bronze_routes: {df_routes.count():,} rows")

In [ ]:
# ── deliveries ───────────────────────────────────────────────────────────────
deliveries_schema = StructType([
    StructField("delivery_id",          StringType(), False),
    StructField("vehicle_id",           StringType(), True),
    StructField("route_id",             StringType(), True),
    StructField("planned_departure",    StringType(), True),
    StructField("actual_arrival",       StringType(), True),
    StructField("planned_duration_hrs", DoubleType(), True),
    StructField("actual_duration_hrs",  DoubleType(), True),
    StructField("delay_hrs",            DoubleType(), True),
    StructField("distance_km",          DoubleType(), True),
    StructField("load_tonnes",          DoubleType(), True),
    StructField("status",               StringType(), True),
    StructField("is_late",              IntegerType(),True),
])

df_deliveries = (
    spark.read.option("header", True).schema(deliveries_schema)
    .csv(f"{RAW_PATH}/deliveries.csv")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_deliveries.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.bronze_deliveries")

print(f"bronze_deliveries: {df_deliveries.count():,} rows")

In [ ]:
# ── fuel_logs ────────────────────────────────────────────────────────────────
fuel_schema = StructType([
    StructField("log_id",          StringType(), False),
    StructField("vehicle_id",      StringType(), True),
    StructField("log_date",        StringType(), True),
    StructField("depot",           StringType(), True),
    StructField("odometer_km",     IntegerType(),True),
    StructField("litres_filled",   DoubleType(), True),
    StructField("cost_per_litre",  DoubleType(), True),
    StructField("total_cost_gbp",  DoubleType(), True),
    StructField("fuel_type",       StringType(), True),
])

df_fuel = (
    spark.read.option("header", True).schema(fuel_schema)
    .csv(f"{RAW_PATH}/fuel_logs.csv")
    .withColumn("log_date",    F.to_date("log_date"))
    .withColumn("_ingested_at", F.current_timestamp())
)

df_fuel.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.bronze_fuel_logs")

print(f"bronze_fuel_logs: {df_fuel.count():,} rows")

In [ ]:
print("\n=== Bronze Ingest Complete ===")
for tbl in ["bronze_vehicles", "bronze_routes", "bronze_deliveries", "bronze_fuel_logs"]:
    cnt = spark.table(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{tbl}").count()
    print(f"  {tbl}: {cnt:,}")